# Semantic Email Clustering with Gemma

This notebook goes beyond keyword labels. Instead of classifying emails by rules, we:
1. Turn each email body into a **vector** (a list of numbers that captures meaning)
2. Group emails that have **similar vectors** into clusters
3. Ask Gemma to **name each cluster** in plain English

This reveals hidden topic threads across your inbox — without reading every email.

**Privacy note:** All processing runs locally via Ollama (`localhost:11434`). No email data leaves your machine.

In [ ]:
# --- Cell 1: Load emails from MultiClass Email Dataset (Kaggle) ---
# Dataset: 2105 synthetic emails across 10 categories
# Fields: subject, body, labels (no sender/date — we add placeholders)

import json
import re
from datetime import datetime, timedelta
import random

DATASET_PATH  = "data/MultiClasssEmail Dataset.json"
EMAIL_SAMPLE_SIZE = 2105   # use all; filter to a smaller number to speed up dev


def clean_text(text):
    if not isinstance(text, str):
        return ""
    return ' '.join(text.split()).strip()

with open(DATASET_PATH) as f:
    raw_data = json.load(f)

# Shuffle for variety then take sample
random.seed(42)
random.shuffle(raw_data)
raw_data = raw_data[:EMAIL_SAMPLE_SIZE]

# Generate evenly-spaced synthetic dates across a plausible year
base_date = datetime(2024, 1, 1)
total_days = 365

emails = []
for i, item in enumerate(raw_data):
    subject = clean_text(item.get("subject", "(no subject)"))
    body    = clean_text(item.get("body", ""))
    if not body or len(body) < 20:
        continue

    labels  = item.get("labels", ["Unknown"])
    # Use the primary category as a pseudo-sender domain
    category = labels[0].lower().replace(" & ", "_").replace(" ", "_")
    sender   = f"no-reply@{category}.example.com"

    # Spread emails across the year
    offset   = timedelta(days=(i / len(raw_data)) * total_days)
    date_iso = (base_date + offset).isoformat()

    emails.append({
        "sender":   sender,
        "subject":  subject,
        "body":     body,
        "date":     date_iso,
        "category": labels[0],   # kept for reference, not used in clustering
    })

print(f"Loaded {len(emails)} emails")

# Category breakdown
from collections import Counter
cats = Counter(e["category"] for e in emails)
for cat, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {n}")


# ── Simulate email sizes and attachments ─────────────────────────────────────
import random as _rnd

CATEGORY_SIZE_KB = {
    "Finance & Bills":       (150, 500,  "PDF"),
    "Travel & Bookings":     (200, 600,  "PDF"),
    "Job Application":       (400, 1200, "DOCX"),
    "Promotions":            (100, 400,  None),
    "Newsletters":           (150, 500,  None),
    "Events & Invitations":  (80,  250,  None),
    "Customer Support":      (60,  200,  None),
    "Business":              (80,  300,  None),
    "Personal":              (50,  150,  None),
    "Reminders":             (50,  120,  None),
}

ATTACHMENT_KEYWORDS = [
    "attached", "attachment", "please find", "enclosed",
    "receipt", "invoice", "statement", "order confirmation",
    "boarding", "itinerary", "booking confirmation", "your ticket",
    "resume", "cv", "cover letter",
]

for i, e in enumerate(emails):
    cat = e.get("category", "Business")
    lo, hi, att = CATEGORY_SIZE_KB.get(cat, (80, 300, None))
    rng      = _rnd.Random(i * 7 + 13)   # seeded per email — reproducible
    base_kb  = rng.uniform(lo, hi)

    # Keyword override — attachment presence from body/subject content
    if att is None:
        text = (e["subject"] + " " + e["body"][:500]).lower()
        for kw in ATTACHMENT_KEYWORDS:
            if kw in text:
                att = "PDF"
                base_kb += rng.uniform(100, 400)
                break

    e["simulated_size_kb"] = round(base_kb, 1)
    e["attachment_type"]   = att   # "PDF" | "DOCX" | None

total_mb = sum(e["simulated_size_kb"] for e in emails) / 1024
print(f"\nSimulated inbox size: {total_mb:.1f} MB across {len(emails)} emails")
att_count = sum(1 for e in emails if e["attachment_type"])
print(f"Emails with attachments: {att_count} ({att_count*100//len(emails)}%)")


In [ ]:
# Load API key from .env file in this folder
from dotenv import load_dotenv
load_dotenv()

# --- Cell 2: LLM helpers ---
# ask_haiku    — Claude Haiku via Anthropic API (cluster labelling, fast)
# ask_gemma    — local Gemma via Ollama (kept for reference)
# get_embedding — local nomic-embed-text via Ollama

import requests
import anthropic as _anthropic

# ── Haiku (Anthropic API) ─────────────────────────────────────────────────────
_haiku_client = _anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

def ask_haiku(prompt, max_tokens=400):
    """Send a prompt to Claude Haiku and return its text response."""
    msg = _haiku_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text.strip()

# ── Gemma / Ollama (local) ────────────────────────────────────────────────────
def ask_gemma(prompt, max_tokens=400):
    """Send a prompt to local Gemma and return its text response."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma4:e2b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0.5, "max_tokens": max_tokens}
        },
        timeout=120
    )
    return response.json()["response"].strip()

def get_embedding(text):
    """Convert text into a vector using the nomic-embed-text model."""
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": text[:3000]},
        timeout=120
    )
    return response.json()["embedding"]

# ── Connection checks ─────────────────────────────────────────────────────────
print("Checking Haiku...")
try:
    print("  Haiku:", ask_haiku("Reply with only: ready", max_tokens=5))
except Exception as e:
    print(f"  Haiku FAILED: {e}")
    print("  → Set ANTHROPIC_API_KEY in the terminal before launching Jupyter.")

print("Checking Ollama (needed for embeddings only)...")
try:
    test_vec = get_embedding("hello world")
    print(f"  Ollama embedding: OK (vector length {len(test_vec)})")
except Exception as e:
    print(f"  Ollama FAILED: {e}")
    print("  → Ollama only needed for Cell 4. Embedding cache can skip it.")


In [ ]:
# --- Cell 3: Cache helpers + initialisation ---
import hashlib, os, json
import numpy as np

EMBED_CACHE_PATH       = "cache/embeddings.npz"
LABEL_CACHE_PATH       = "cache/labels.json"        # Gemma label cache
HAIKU_LABEL_CACHE_PATH = "cache/labels_haiku.json"  # Haiku label cache (separate)
HAIKU_MAX_WORKERS      = 4    # keep low to avoid hitting token-per-minute rate limits

def load_embed_cache():
    if os.path.exists(EMBED_CACHE_PATH):
        data = np.load(EMBED_CACHE_PATH, allow_pickle=True)
        return dict(data["cache"].item())
    return {}

def save_embed_cache(cache):
    os.makedirs("cache", exist_ok=True)
    np.savez_compressed(EMBED_CACHE_PATH, cache=np.array(cache, dtype=object))

def get_embedding_cached(text, cache):
    """Return cached embedding or call Ollama and store result."""
    key = hashlib.sha256(text[:3000].encode()).hexdigest()
    if key not in cache:
        cache[key] = get_embedding(text)
    return cache[key]

def load_label_cache():
    if os.path.exists(LABEL_CACHE_PATH):
        with open(LABEL_CACHE_PATH) as f:
            return json.load(f)
    return {}

def save_label_cache(cache):
    os.makedirs("cache", exist_ok=True)
    with open(LABEL_CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)

def load_haiku_label_cache():
    if os.path.exists(HAIKU_LABEL_CACHE_PATH):
        with open(HAIKU_LABEL_CACHE_PATH) as f:
            return json.load(f)
    return {}

def save_haiku_label_cache(cache):
    os.makedirs("cache", exist_ok=True)
    with open(HAIKU_LABEL_CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)

def cluster_cache_key(email_ids):
    return hashlib.sha256("|".join(sorted(str(i) for i in email_ids)).encode()).hexdigest()

embed_cache       = load_embed_cache()
label_cache       = load_label_cache()
haiku_label_cache = load_haiku_label_cache()

print(f"Embedding cache:    {len(embed_cache)} entries loaded.")
print(f"Gemma label cache:  {len(label_cache)} entries loaded.")
print(f"Haiku label cache:  {len(haiku_label_cache)} entries loaded.")
if embed_cache:
    print(f"  → up to {len(embed_cache)} emails will skip Ollama embedding calls.")


In [ ]:
# --- Cell 4: Cluster emails with DBSCAN ---
#
# DBSCAN groups nearby emails together and avoids "chaining" issues.
# We also compute cluster centroids so we can find semantic links between clusters later.
#
# Tune these numbers to change cluster sizes:
DBSCAN_EPS = 0.38       # Distance threshold (lower = tighter clusters)
DBSCAN_MIN_SAMPLES = 3     # Minimum emails needed to form a cluster

import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize

print(f"Building embeddings for {len(emails)} emails...")
print(f"(Cache has {len(embed_cache)} entries; cached emails skip Ollama)\n")

vectors = []
for i, e in enumerate(emails):
    if i % 50 == 0:
        print(f"  {i}/{len(emails)}")
    text_for_embedding = f"Subject: {e['subject']}\n\nBody: {e['body']}"
    vectors.append(get_embedding_cached(text_for_embedding, embed_cache))

# Stack all vectors into a 2D array: one row = one email
X = np.array(vectors, dtype=np.float32)
save_embed_cache(embed_cache)
print(f"\nEmbedding matrix shape: {X.shape}")
print("(rows = emails, columns = embedding dimensions)\n")

# Normalize vectors
X_norm = normalize(X, norm='l2')

# Run DBSCAN clustering
dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric='euclidean')
cluster_labels = dbscan.fit_predict(X_norm)

# Attach cluster IDs to emails
for i, e in enumerate(emails):
    e["cluster_id"] = int(cluster_labels[i])

# Build cluster centroids (average vector for each cluster)
cluster_centroids = {}
unique_clusters = sorted([c for c in set(cluster_labels) if c != -1])
for cid in unique_clusters:
    cluster_vectors = X_norm[cluster_labels == cid]
    centroid = cluster_vectors.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-12)  # normalize centroid
    cluster_centroids[int(cid)] = centroid

# Print summary
from collections import Counter
counts = Counter(cluster_labels)
n_clusters = len([c for c in counts if c != -1])
n_noise = counts.get(-1, 0)

print(f"Found {n_clusters} clusters, {n_noise} outlier emails\n")
for cid in sorted(c for c in counts if c != -1):
    print(f"  Cluster {cid}: {counts[cid]} emails")

if n_noise > 0:
    print(f"  (Outliers): {n_noise} emails")

In [ ]:
# --- Cell 4b: Split oversized sub-clusters (max 12 emails each) ---
from sklearn.cluster import KMeans
from math import ceil
import numpy as np

MAX_SC_SIZE = 12

# ── Split large sub-clusters ──────────────────────────────────────────────────
next_cid   = int(max(cluster_centroids.keys())) + 1
new_labels = cluster_labels.copy()

for cid in sorted(cluster_centroids.keys()):
    members = np.where(cluster_labels == cid)[0]
    if len(members) <= MAX_SC_SIZE:
        continue
    n_parts  = ceil(len(members) / MAX_SC_SIZE)
    km       = KMeans(n_clusters=n_parts, random_state=42, n_init="auto")
    part_lbl = km.fit_predict(X_norm[members])
    for part in range(1, n_parts):
        for idx in members[part_lbl == part]:
            new_labels[idx] = next_cid
        next_cid += 1
    print(f"  Split cluster {cid} ({len(members)} emails) → {n_parts} parts of ≤{MAX_SC_SIZE}")

cluster_labels = new_labels
for i, e in enumerate(emails):
    e["cluster_id"] = int(cluster_labels[i])

# ── Rebuild centroids for ALL clusters (originals + new split ones) ───────────
cluster_centroids = {}
unique_clusters   = sorted(set(cluster_labels) - {-1})
for cid in unique_clusters:
    vecs     = X_norm[cluster_labels == cid]
    centroid = vecs.mean(axis=0)
    cluster_centroids[int(cid)] = centroid / (np.linalg.norm(centroid) + 1e-12)

# ── Assign noise emails to nearest sub-cluster centroid ───────────────────────
noise_idx = np.where(cluster_labels == -1)[0]
if len(noise_idx) > 0:
    centroid_vecs = np.array([cluster_centroids[c] for c in unique_clusters])
    sims          = X_norm[noise_idx] @ centroid_vecs.T
    best          = np.argmax(sims, axis=1)
    for ni, bi in zip(noise_idx, best):
        cluster_labels[ni]       = unique_clusters[bi]
        emails[ni]["cluster_id"] = int(unique_clusters[bi])
    # Final centroid rebuild after noise reassignment
    unique_clusters = sorted(set(cluster_labels) - {-1})
    for cid in unique_clusters:
        vecs     = X_norm[cluster_labels == cid]
        centroid = vecs.mean(axis=0)
        cluster_centroids[int(cid)] = centroid / (np.linalg.norm(centroid) + 1e-12)
    print(f"  Assigned {len(noise_idx)} noise emails to nearest sub-cluster")

from collections import Counter
counts = Counter(cluster_labels)
print(f"\nAfter split: {len(unique_clusters)} sub-clusters")
print(f"  Largest: {max(counts[c] for c in unique_clusters)} emails")
print(f"  Average: {sum(counts[c] for c in unique_clusters)/len(unique_clusters):.1f} emails")
print(f"  All ≤{MAX_SC_SIZE}: {all(counts[c] <= MAX_SC_SIZE for c in unique_clusters)}")


In [ ]:
# --- Cell 5: Label clusters via Haiku + build cluster similarity links ---
#
# name_cluster_haiku uses the same full-content prompt as the Gemma version
# but calls Haiku (~1s/call vs ~25s). Serial execution avoids rate-limit crashes.
# Default values are set before every API call so UnboundLocalError cannot occur.

import json
import numpy as np
import re
import time

CLUSTER_LINK_THRESHOLD = 0.70
cluster_title_lookup = {cid: f"Cluster {cid}" for cid in sorted(cluster_centroids.keys())}


def finish_sentence(text):
    text = str(text).strip()
    if not text:
        return text
    matches = list(re.finditer(r'[.!?]', text))
    if matches:
        text = text[:matches[-1].end()].strip()
    if text and text[-1] not in '.!?':
        text += '.'
    return text


def name_cluster_haiku(cluster_emails, related_cluster_ids=None):
    """Label a cluster via Haiku — full email content, same quality as Gemma."""
    if related_cluster_ids is None:
        related_cluster_ids = []
    if len(cluster_emails) <= 6:
        sample = cluster_emails
    else:
        sample = sorted(cluster_emails, key=lambda e: len(e.get('body', '')), reverse=True)[:5]

    examples = "\n".join(
        f"- Subject: {e['subject'][:80]} | Body: {e['body'][:300]}"
        for e in sample
    )
    related_text = "\n".join(
        f"  - Cluster {cid}: {cluster_title_lookup.get(cid, f'Cluster {cid}')}"
        for cid in related_cluster_ids[:5]
    )

    prompt = f"""You are labelling a SPECIFIC sub-cluster of emails within a larger group.
These emails already share a general category with neighbouring clusters.
Your job is to find what makes THIS cluster distinct and specific — not the general theme.

First reason: what SPECIFIC detail distinguishes these emails from other similar ones?
Look for: destinations, company names, product names, amounts, dates, people, reference numbers.

Then return ONLY valid JSON:
{{
  "reasoning": "1-2 sentences: what specific detail distinguishes this cluster?",
  "title": "short title (max 6-7 words) — use the SPECIFIC distinguishing detail, not the general category",
  "summary": "one sentence naming the specific route/product/person/company/amount involved"
}}

Rules:
- NEVER use a generic category as the title (e.g., not \"Flight Booking Confirmations\", not \"Meeting Reminders\").
- Use the most specific detail available: route, destination, company, person, product, amount.
- If emails share the same subject template, look inside the body for what differs between them.
- If all emails are genuinely identical with no distinguishing detail, say so honestly.

BAD title: \"Flight Booking Confirmations\"  → too generic
GOOD title: \"New York to London Flight\"    → specific destination

BAD title: \"Meeting Reminder Emails\"        → too generic
GOOD title: \"Q3 Sales Review Meetings\"      → specific meeting type

BAD title: \"Order Confirmation Emails\"      → too generic
GOOD title: \"Amazon Electronics Orders\"     → specific merchant + category

Related clusters (for context only):
{related_text}

Email samples:
{examples}"""

    response = ask_haiku(prompt, max_tokens=400)
    start = response.find("{")
    end   = response.rfind("}") + 1
    if start >= 0 and end > start:
        data = json.loads(response[start:end])
        return (
            str(data.get("title",     "Unclassified")).strip()[:60],
            finish_sentence(str(data.get("summary",   "No summary.")))[:220],
            finish_sentence(str(data.get("reasoning", "")))[:400],
        )
    return "Unclassified", finish_sentence("Could not parse response."), ""


# ── Build cluster similarity links ────────────────────────────────────────────
cluster_links = []
cluster_ids   = sorted(cluster_centroids.keys())

for i, cid_a in enumerate(cluster_ids):
    for cid_b in cluster_ids[i + 1:]:
        sim = float(np.dot(cluster_centroids[cid_a], cluster_centroids[cid_b]))
        if sim >= CLUSTER_LINK_THRESHOLD:
            cluster_links.append({
                "cluster_a": cid_a, "cluster_b": cid_b, "similarity": round(sim, 4)
            })

linked_neighbors = {cid: [] for cid in cluster_ids}
for link in cluster_links:
    linked_neighbors[link["cluster_a"]].append(link["cluster_b"])
    linked_neighbors[link["cluster_b"]].append(link["cluster_a"])


# ── Serial Haiku labelling with cache ─────────────────────────────────────────
cluster_profiles = []
print(f"Labelling {len(cluster_ids)} clusters via Haiku (full content)...")

cached_count = sum(
    1 for cid in cluster_ids
    if cluster_cache_key([i for i, e in enumerate(emails) if e["cluster_id"] == cid])
    in haiku_label_cache
)
print(f"  Cache hits: {cached_count} | Haiku calls needed: {len(cluster_ids) - cached_count}\n")

all_labels = {}

for n, cid in enumerate(cluster_ids, 1):
    group     = [e for e in emails if e["cluster_id"] == cid]
    email_ids = [i for i, e in enumerate(emails) if e["cluster_id"] == cid]
    key       = cluster_cache_key(email_ids)
    neighbors = sorted(linked_neighbors.get(cid, []))

    # Always initialise — prevents UnboundLocalError if API fails
    title, summary, reasoning = "Unclassified", finish_sentence("Could not label."), ""

    if key in haiku_label_cache:
        d = haiku_label_cache[key]
        title, summary, reasoning = d["title"], d["summary"], d.get("reasoning", "")
        print(f"  [{n}/{len(cluster_ids)}] Cluster {cid} [cached]: {title}")
    else:
        t0 = time.time()
        try:
            title, summary, reasoning = name_cluster_haiku(group, related_cluster_ids=neighbors)
            haiku_label_cache[key] = {"title": title, "summary": summary, "reasoning": reasoning}
            save_haiku_label_cache(haiku_label_cache)
            print(f"  [{n}/{len(cluster_ids)}] Cluster {cid} ({time.time()-t0:.1f}s): {title}")
            if reasoning:
                print(f"    ↳ {reasoning[:110]}")
        except Exception as e:
            print(f"  [{n}/{len(cluster_ids)}] Cluster {cid} ERROR ({type(e).__name__}): {e}")

    cluster_title_lookup[cid] = title
    all_labels[cid] = (title, summary, reasoning)


print("\nBuilding cluster profiles...\n")
for cid in cluster_ids:
    title, summary, reasoning = all_labels[cid]
    neighbors = sorted(linked_neighbors.get(cid, []))
    if neighbors:
        links_text = ", ".join(str(x) for x in neighbors[:5])
        narrative_text = (
            f"{title} focuses on {summary.lower().rstrip('.')} "
            f"and connects semantically with clusters {links_text}."
        )
    else:
        narrative_text = (
            f"{title} focuses on {summary.lower().rstrip('.')} "
            f"and forms a mostly standalone semantic theme."
        )
    cluster_profiles.append({
        "cluster_id":      cid,
        "email_count":     len([e for e in emails if e["cluster_id"] == cid]),
        "title":           title,
        "summary":         summary,
        "reasoning":       reasoning,
        "narrative":       finish_sentence(narrative_text),
        "linked_clusters": ",".join(str(x) for x in neighbors),
    })

print(f"\nDone. Created {len(cluster_profiles)} semantic cluster profiles.")
print(f"Found {len(cluster_links)} cluster-to-cluster semantic links.")


In [ ]:
# --- Cell 6: Save results to CSV files ---

import os
import pandas as pd

os.makedirs("output/email_semantics", exist_ok=True)

# Save email-level data (each email + assigned primary cluster)
emails_df = pd.DataFrame([
    {
        "sender":       e["sender"],
        "subject":      e["subject"],
        "body":         e["body"],
        "body_preview": e["body"][:200],
        "cluster_id":   e["cluster_id"],
        "date":         e.get("date"),
    }
    for e in emails
])
emails_df.to_csv("output/email_semantics/emails_with_clusters_haiku.csv", index=False)

# Save semantic cluster profiles
clusters_df = pd.DataFrame(cluster_profiles)
clusters_df.to_csv("output/email_semantics/cluster_labels_haiku.csv", index=False)

# Save cluster-to-cluster semantic links (graph edges)
links_df = pd.DataFrame(cluster_links)
links_df.to_csv("output/email_semantics/cluster_links_haiku.csv", index=False)

print("Saved:")
print("  output/email_semantics/emails_with_clusters_haiku.csv")
print("  output/email_semantics/cluster_labels_haiku.csv")
print("  output/email_semantics/cluster_links_haiku.csv")
print(f"\nTotal: {len(emails)} emails across {len(cluster_profiles)} clusters")
print(f"Semantic links between clusters: {len(cluster_links)}")
dates_in_df = emails_df["date"].notna().sum()
print(f"Emails with date: {dates_in_df}/{len(emails_df)}")

In [ ]:
# --- Cell 7: Build sub-cluster descriptor dataframe ---
# For each of the 64 sub-clusters, collect metadata + a cohesion score
# (avg cosine distance of member emails from their centroid).
# Also builds centroids_matrix (64×768) needed by the next cell.

from sklearn.metrics.pairwise import cosine_distances

# emails_df was built fresh in Cell 6 with a 0-based integer index that
# aligns directly with X_norm rows, so no reset_index needed.

rows = []
centroids_list = []  # will become centroids_matrix, order = sorted cluster_id

for cid in sorted(cluster_centroids.keys()):
    row = clusters_df[clusters_df["cluster_id"] == cid].iloc[0]

    mask = emails_df["cluster_id"] == cid
    cluster_emails = emails_df[mask]

    # Top 3 most frequent senders
    top_senders = cluster_emails["sender"].value_counts().head(3).index.tolist()

    # Centroid (already normalized)
    centroid = cluster_centroids[cid]
    centroids_list.append(centroid)

    # Cohesion: average cosine distance of member emails from their centroid
    email_indices = emails_df.index[mask].tolist()
    cluster_vecs = X_norm[email_indices]
    dists = cosine_distances(cluster_vecs, centroid.reshape(1, -1)).flatten()
    cohesion = float(np.mean(dists))

    rows.append({
        "cluster_id": cid,
        "title": row["title"],
        "summary": row["summary"],
        "email_count": int(row["email_count"]),
        "top_senders": ", ".join(top_senders),
        "cohesion_score": round(cohesion, 4),
    })

sub_cluster_descriptors = pd.DataFrame(rows).sort_values("cluster_id").reset_index(drop=True)
centroids_matrix = np.array(centroids_list)  # shape (64, 768)

sub_cluster_descriptors.to_csv("output/email_semantics/sub_cluster_descriptors.csv", index=False)
print(f"Saved sub_cluster_descriptors.csv — {len(sub_cluster_descriptors)} rows")
display(sub_cluster_descriptors)

In [ ]:
# --- Cell 8: Compute macro-clusters from sub-cluster centroids ---
# Run HDBSCAN on the 64 centroid vectors to find 8-12 macro-clusters.
# Noise sub-clusters (-1) are reassigned to nearest macro-cluster by cosine similarity.

import hdbscan
from sklearn.metrics.pairwise import cosine_similarity

# centroids_matrix rows are sorted by cluster_id (same order as sub_cluster_descriptors)
macro_labels = None
best_mcs = None

for mcs in range(2, 7):  # try min_cluster_size 2 through 6
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=2,
        metric="euclidean",  # normalized vecs → euclidean ≈ cosine distance
    )
    labels = clusterer.fit_predict(centroids_matrix)
    n_macro = len(set(labels) - {-1})
    print(f"min_cluster_size={mcs} → {n_macro} macro-clusters, {(labels == -1).sum()} noise sub-clusters")
    if 8 <= n_macro <= 12:
        macro_labels = labels.copy()
        best_mcs = mcs
        break

if macro_labels is None:
    print("\nNo parameter hit 8–12 range; using last result.")
    macro_labels = labels.copy()

print(f"\nUsing min_cluster_size={best_mcs}")

# Assign noise sub-clusters to nearest macro-cluster by cosine similarity
valid_macro_ids = sorted(set(macro_labels) - {-1})
macro_centroids_agg = np.array([
    centroids_matrix[macro_labels == m].mean(axis=0) for m in valid_macro_ids
])

final_macro_labels = macro_labels.copy()
for i, lbl in enumerate(macro_labels):
    if lbl == -1:
        sims = cosine_similarity(centroids_matrix[i].reshape(1, -1), macro_centroids_agg)[0]
        final_macro_labels[i] = valid_macro_ids[int(np.argmax(sims))]

sub_cluster_descriptors["macro_cluster_id"] = final_macro_labels

print(f"\nFinal macro-cluster membership ({len(set(final_macro_labels))} macro-clusters):")
for m in sorted(set(final_macro_labels)):
    members = sub_cluster_descriptors[sub_cluster_descriptors["macro_cluster_id"] == m]["cluster_id"].tolist()
    print(f"  Macro {m}: sub-clusters {members}")

In [ ]:
# --- Cell 9: Ask Haiku to label each macro-cluster (with label cache) ---
# For each macro-cluster, pass the titles + summaries of its sub-clusters
# and ask for a macro-level title (max 5 words) and one-sentence summary.

macro_cluster_labels = []

for m in sorted(set(final_macro_labels)):
    group      = sub_cluster_descriptors[sub_cluster_descriptors["macro_cluster_id"] == m]
    member_ids = sorted(group["cluster_id"].tolist())
    cache_key  = cluster_cache_key([f"macro_v1_{i}" for i in member_ids])

    if cache_key in label_cache:
        d       = label_cache[cache_key]
        title   = d["title"]
        summary = d["summary"]
        print(f"Macro {m} (cached): {title}")
    else:
        sub_descriptions = "\n".join(
            f"- {r['title']}: {r['summary']}"
            for _, r in group.iterrows()
        )
        prompt = f"""You are labeling a group of related email topic clusters.

The following sub-clusters have been grouped together:
{sub_descriptions}

Provide:
1. A macro-level title (max 5 words) that captures the common theme
2. A one-sentence summary describing what connects these sub-clusters

Respond ONLY in valid JSON (no markdown, no extra text):
{{"title": "...", "summary": "..."}}"""

        response = ask_haiku(prompt, max_tokens=300)
        clean = re.sub(r"```json|```", "", response).strip()
        start = clean.find("{")
        end   = clean.rfind("}") + 1
        try:
            parsed  = json.loads(clean[start:end])
            title   = str(parsed.get("title", "Unlabeled")).strip()[:60]
            summary = str(parsed.get("summary", "")).strip()
        except (json.JSONDecodeError, ValueError):
            lines   = clean.strip().splitlines()
            title   = lines[0][:60] if lines else "Unlabeled"
            summary = lines[1] if len(lines) > 1 else ""

        label_cache[cache_key] = {"title": title, "summary": summary}
        save_label_cache(label_cache)
        print(f"Macro {m}: {title}")

    macro_cluster_labels.append({
        "macro_cluster_id":   m,
        "title":              title,
        "summary":            summary,
        "sub_cluster_count":  len(group),
        "total_emails":       int(group["email_count"].sum()),
        "sub_cluster_ids":    list(group["cluster_id"]),
        "sub_cluster_titles": list(group["title"]),
    })

macro_labels_df = pd.DataFrame(macro_cluster_labels)
print(f"\nLabeled {len(macro_labels_df)} macro-clusters.")


In [ ]:
# --- Cell 10: Save analysis CSVs and print hierarchy summary ---

# macro_cluster_labels_haiku.csv — list columns serialised as JSON strings for CSV safety
macro_labels_csv = macro_labels_df.copy()
macro_labels_csv["sub_cluster_ids"] = macro_labels_csv["sub_cluster_ids"].apply(json.dumps)
macro_labels_csv["sub_cluster_titles"] = macro_labels_csv["sub_cluster_titles"].apply(json.dumps)
macro_labels_csv.to_csv("output/email_semantics/macro_cluster_labels_haiku.csv", index=False)

# sub_cluster_descriptors_with_macro_haiku.csv — all descriptor columns + macro info
macro_title_map = dict(zip(macro_labels_df["macro_cluster_id"], macro_labels_df["title"]))
sub_cluster_descriptors["macro_cluster_title"] = sub_cluster_descriptors["macro_cluster_id"].map(macro_title_map)
sub_cluster_descriptors.to_csv("output/email_semantics/sub_cluster_descriptors_with_macro_haiku.csv", index=False)

# Print readable hierarchy
print("\n" + "=" * 70)
print("MACRO-CLUSTER HIERARCHY")
print("=" * 70)
for _, macro_row in macro_labels_df.sort_values("macro_cluster_id").iterrows():
    print(f"\nMACRO {macro_row['macro_cluster_id']}: {macro_row['title']}")
    print(f"  {macro_row['summary']}")
    print(f"  {macro_row['sub_cluster_count']} sub-clusters | {macro_row['total_emails']} emails")
    sub_rows = sub_cluster_descriptors[
        sub_cluster_descriptors["macro_cluster_id"] == macro_row["macro_cluster_id"]
    ][["cluster_id", "title", "email_count"]].sort_values("cluster_id")
    for _, sr in sub_rows.iterrows():
        print(f"    • [{sr['cluster_id']}] {sr['title']} ({sr['email_count']} emails)")
print("\n" + "=" * 70)
print("Saved:")
print("  output/email_semantics/macro_cluster_labels_haiku.csv")
print("  output/email_semantics/sub_cluster_descriptors_with_macro_haiku.csv")

# UMAP → HDBSCAN → AI Macro-Clustering Pipeline (v2)

This pipeline improves on the earlier macro-clustering by:
1. **UMAP** — compress 768-dim sub-cluster centroids to 8 dims before clustering
2. **HDBSCAN** — cluster in the low-dim UMAP space (better geometry than raw cosine)
3. **AI labelling** — Gemma names each macro-cluster and places noise sub-clusters

In [ ]:
# --- Cell 11: STEP 1 — UMAP dimensionality reduction on sub-cluster centroids ---
# Compress centroids_matrix (64×768) → (64×8) before clustering.
# UMAP finds a low-dimensional manifold that preserves local neighbourhood structure
# far better than PCA, giving HDBSCAN cleaner geometric boundaries to cut.

import subprocess, importlib, sys
import numpy as np
import umap
from sklearn.preprocessing import normalize

# centroids_matrix was built in Cell 7 (shape: n_subclusters × 768).
# Rows are ordered by ascending cluster_id, matching sub_cluster_descriptors.
print(f"Input centroid matrix shape: {centroids_matrix.shape}")

UMAP_N_NEIGHBORS  = 20
UMAP_N_COMPONENTS = 8
UMAP_MIN_DIST     = 0.1

reducer = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=UMAP_N_COMPONENTS,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",      # cosine is natural for normalised embedding vectors
    random_state=42,
 )

umap_matrix = reducer.fit_transform(centroids_matrix)   # shape: (64, 8)

print(f"UMAP output shape: {umap_matrix.shape}")
print(f"Parameters: n_neighbors={UMAP_N_NEIGHBORS}, n_components={UMAP_N_COMPONENTS}, "
      f"min_dist={UMAP_MIN_DIST}")
print("STEP 1 complete — UMAP matrix ready for HDBSCAN.")

In [ ]:
# --- Cell 12: STEP 2 — HDBSCAN on UMAP output with parameter search ---
# Target: 4–8 macro-clusters.
# If the default UMAP params miss, retry across a grid of n_neighbors / n_components.

import hdbscan
import itertools

VALID_MIN = 5
VALID_MAX = 30

HDBSCAN_MIN_CLUSTER_SIZE = 2
HDBSCAN_MIN_SAMPLES      = 2

# Grid to search if initial run is outside [4, 8]
NEIGHBOR_GRID    = [10, 15, 20, 30, 40]
COMPONENT_GRID   = [5, 8, 10]

def run_hdbscan(matrix):
    """Run HDBSCAN with fixed hyperparameters; return labels array."""
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
    )
    return clusterer.fit_predict(matrix)

# --- Initial attempt with the UMAP matrix already computed ---
print("=" * 60)
print("HDBSCAN parameter search")
print("=" * 60)
print(f"\nInitial attempt  n_neighbors={UMAP_N_NEIGHBORS}  n_components={UMAP_N_COMPONENTS}")
macro_raw = run_hdbscan(umap_matrix)
n_macro   = len(set(macro_raw) - {-1})
n_noise   = int((macro_raw == -1).sum())
print(f"  → {n_macro} macro-clusters, {n_noise} noise sub-clusters")

found_valid   = (VALID_MIN <= n_macro <= VALID_MAX)
final_labels  = macro_raw
final_nn      = UMAP_N_NEIGHBORS
final_nc      = UMAP_N_COMPONENTS
final_matrix  = umap_matrix

# --- Grid search if needed ---
best_n_macro = len(set(final_labels) - {-1})  # track best result across all attempts
best_labels  = final_labels.copy()
best_matrix  = final_matrix

if not found_valid:
    print(f"\nResult outside [{VALID_MIN}, {VALID_MAX}] — starting grid search...\n")
    for nn, nc in itertools.product(NEIGHBOR_GRID, COMPONENT_GRID):
        if nn == UMAP_N_NEIGHBORS and nc == UMAP_N_COMPONENTS:
            continue
        r = umap.UMAP(
            n_neighbors=nn,
            n_components=nc,
            min_dist=UMAP_MIN_DIST,
            metric="cosine",
            random_state=42,
        ).fit_transform(centroids_matrix)
        labels  = run_hdbscan(r)
        n_m     = len(set(labels) - {-1})
        n_ns    = int((labels == -1).sum())
        print(f"  n_neighbors={nn:2d}  n_components={nc}  → {n_m} macro-clusters, {n_ns} noise")
        # Keep track of the result closest to target range (most non-noise clusters)
        if n_m > best_n_macro:
            best_n_macro = n_m
            best_labels  = labels.copy()
            best_matrix  = r
        if VALID_MIN <= n_m <= VALID_MAX:
            found_valid  = True
            final_labels = labels
            final_nn     = nn
            final_nc     = nc
            final_matrix = r
            break
    # If still no valid result, use best seen rather than last attempt
    if not found_valid:
        final_labels = best_labels
        final_matrix = best_matrix
        print(f"  → Using best result found: {best_n_macro} macro-clusters")

print()
if found_valid:
    print(f"✓ Valid result found.")
else:
    print("⚠  No combination produced a result in [4, 8]. Using best available result.")

print(f"\nFinal parameters used: n_neighbors={final_nn}, n_components={final_nc}, "
      f"min_dist={UMAP_MIN_DIST}")
n_final_macro = len(set(final_labels) - {-1})
n_final_noise = int((final_labels == -1).sum())
print(f"Final result: {n_final_macro} macro-clusters, {n_final_noise} noise sub-clusters")

# Split into assigned and noise
noise_indices    = [i for i, l in enumerate(final_labels) if l == -1]
assigned_indices = [i for i, l in enumerate(final_labels) if l != -1]
valid_macro_ids  = sorted(set(final_labels) - {-1})

print(f"\nNoise sub-clusters (will be placed by AI): "
      f"{[sub_cluster_descriptors.iloc[i]['cluster_id'] for i in noise_indices]}")

In [ ]:
# --- Cell 12c: Split oversized macros, merge tiny ones ---
# Inserts between Cell 12 (HDBSCAN) and Cell 13 (AI labelling).
# Uses final_matrix (UMAP positions per sub-cluster) and final_labels.
# Rewrites final_labels and valid_macro_ids so Cell 13 and Cell 16
# both work unchanged.

from math import ceil
from sklearn.cluster import KMeans
import numpy as np

MAX_SC = 15   # split any macro with more sub-clusters than this
MIN_SC = 3    # merge any macro with fewer sub-clusters than this

# Build macro_id → list of positions in final_matrix (one row per sub-cluster)
sc_by_macro = {}
for pos, mid in enumerate(final_labels):
    if int(mid) == -1:
        continue
    sc_by_macro.setdefault(int(mid), []).append(pos)

print(f"Before split/merge: {len(sc_by_macro)} macros")
print(f"Largest: {max(len(v) for v in sc_by_macro.values())} sub-clusters")
print()

# ── Split ─────────────────────────────────────────────────────────────────────
next_id = int(max(sc_by_macro.keys())) + 1

for mid in list(sc_by_macro.keys()):
    sc_rows = sc_by_macro[mid]
    if len(sc_rows) <= MAX_SC:
        continue
    n_parts  = ceil(len(sc_rows) / MAX_SC)
    sc_vecs  = final_matrix[sc_rows]
    km       = KMeans(n_clusters=n_parts, random_state=42, n_init="auto")
    part_lbl = km.fit_predict(sc_vecs)
    for part in range(n_parts):
        part_rows = [sc_rows[i] for i, l in enumerate(part_lbl) if l == part]
        if part == 0:
            sc_by_macro[mid] = part_rows
        else:
            sc_by_macro[next_id] = part_rows
            next_id += 1
    print(f"  Split macro {mid} ({len(sc_rows)} subs) → {n_parts} parts of ~{MAX_SC}")

# ── Merge tiny macros ─────────────────────────────────────────────────────────
for mid in list(sc_by_macro.keys()):
    sc_rows = sc_by_macro.get(mid, [])
    if len(sc_rows) >= MIN_SC or not sc_rows:
        continue
    my_vec = final_matrix[sc_rows].mean(axis=0)
    best_sim, best_mid = -1, None
    for other_mid, other_rows in sc_by_macro.items():
        if other_mid == mid or not other_rows:
            continue
        other_vec = final_matrix[other_rows].mean(axis=0)
        denom = (np.linalg.norm(my_vec) * np.linalg.norm(other_vec) + 1e-9)
        sim   = float(np.dot(my_vec, other_vec) / denom)
        if sim > best_sim:
            best_sim, best_mid = sim, other_mid
    if best_mid is not None:
        sc_by_macro[best_mid].extend(sc_rows)
        del sc_by_macro[mid]
        print(f"  Merged tiny macro {mid} ({len(sc_rows)} subs) → macro {best_mid}")

# ── Rewrite final_labels so later cells work unchanged ────────────────────────
updated = np.full(len(final_labels), -1, dtype=int)
for mid, sc_rows in sc_by_macro.items():
    for pos in sc_rows:
        updated[pos] = mid
final_labels    = updated
valid_macro_ids = sorted(sc_by_macro.keys())

print(f"\nAfter split/merge: {len(valid_macro_ids)} macros")
print(f"Largest: {max(len(sc_by_macro[m]) for m in valid_macro_ids)} sub-clusters")
print(f"All ≤{MAX_SC}: {all(len(sc_by_macro[m]) <= MAX_SC for m in valid_macro_ids)}")


In [ ]:
# --- Cell 13: STEP 3 — AI labelling and noise placement (Haiku) ---
# Part A: label each HDBSCAN macro-cluster via Gemma.
# Part B: for each noise sub-cluster, ask Gemma which macro it belongs to.

import json, re

def parse_json_response(response):
    """Extract the first JSON object from a Gemma response string."""
    clean = re.sub(r"```json|```", "", response).strip()
    start = clean.find("{")
    end   = clean.rfind("}") + 1
    if start >= 0 and end > start:
        return json.loads(clean[start:end])
    raise ValueError("No JSON object found in response")


# ── Pre-processing: build macro → emails lookup and entity helpers ────────────

import re
from collections import Counter

def extract_key_entities(subjects, min_count=2):
    """Extract capitalised tokens appearing 2+ times across subjects."""
    all_tokens = []
    for s in subjects:
        all_tokens.extend(re.findall(r'\b[A-Z][a-zA-Z]{2,}\b', s))
    stopwords = {'Re', 'Fw', 'Fwd', 'The', 'This', 'For', 'From', 'Please', 'Email', 'Subject', 'New', 'Update', 'Info'}
    counts = Counter(t for t in all_tokens if t not in stopwords)
    return [word for word, cnt in counts.most_common(8) if cnt >= min_count]

# Build macro_id → list of sub-cluster indices using final_labels
macro_to_sc_indices = {}
for idx, lbl in enumerate(final_labels):
    if lbl != -1:
        macro_to_sc_indices.setdefault(lbl, []).append(idx)

def macro_context(mid):
    """Return (top_senders_str, date_range_str, entity_list) for a macro-cluster."""
    sc_indices = macro_to_sc_indices.get(mid, [])
    sc_ids = [int(sub_cluster_descriptors.iloc[i]['cluster_id']) for i in sc_indices]

    # Aggregate emails for this macro
    macro_emails = emails_df[emails_df['cluster_id'].isin(sc_ids)]

    # Top 3 senders
    top_senders = macro_emails['sender'].value_counts().head(3).index.tolist()
    sender_str = ', '.join(top_senders) if top_senders else 'various'

    # Date range
    dates = macro_emails['date'].dropna().tolist()
    date_range = f"{min(dates)} to {max(dates)}" if dates else 'unknown'

    # Key entities from subjects
    subjects = macro_emails['subject'].dropna().tolist()
    entities = extract_key_entities(subjects)
    entity_str = ', '.join(entities) if entities else 'none identified'

    return sender_str, date_range, entity_str


# ── Part A: label assigned macro-clusters ────────────────────────────────────

macro_labels_v2 = {}   # macro_cluster_id → {title, summary, ...}

print("Labelling macro-clusters...\n")
for mid in valid_macro_ids:
    member_rows = [
        sub_cluster_descriptors.iloc[i]
        for i, l in enumerate(final_labels)
        if l == mid
    ]
    member_sc_ids = sorted(int(r["cluster_id"]) for r in member_rows)
    cache_key     = cluster_cache_key([f"macro_v2_{i}" for i in member_sc_ids])

    if cache_key in label_cache:
        d        = label_cache[cache_key]
        title    = d["title"]
        summary  = d["summary"]
        coherent = d.get("coherent", True)
        print(f"Macro {mid} (cached): {title}")
    else:
        sub_descriptions = "\n".join(
            f"- {r['title']}: {r['summary']}" for r in member_rows
        )
        sender_str, date_range, entity_str = macro_context(mid)

        prompt = f"""You are describing a region of someone's inbox.

Key entities that appear repeatedly across email subjects: {entity_str}
Most active senders in this region: {sender_str}
Date range: {date_range}

Sub-topics within this region:
{sub_descriptions}

Return ONLY valid JSON (no markdown, no extra text):
{{
  "title": "max 5 words, name a specific project or topic",
  "summary": "2-3 sentences naming the specific projects, people, or organisations involved — not just the general theme",
  "coherent": true
}}

If the grouping seems incoherent set "coherent": false but still provide a title and summary."""

        response = ask_haiku(prompt, max_tokens=300)
        try:
            parsed   = parse_json_response(response)
            title    = str(parsed.get("title", "Unlabeled")).strip()[:60]
            summary  = str(parsed.get("summary", "")).strip()
            coherent = bool(parsed.get("coherent", True))
        except (json.JSONDecodeError, ValueError):
            lines    = response.strip().splitlines()
            title    = lines[0][:60] if lines else "Unlabeled"
            summary  = lines[1] if len(lines) > 1 else ""
            coherent = True

        label_cache[cache_key] = {"title": title, "summary": summary, "coherent": coherent}
        save_label_cache(label_cache)
        if not coherent:
            print(f"  ⚠ WARNING: Haiku flagged macro {mid} as incoherent — keeping anyway.")
        print(f"Macro {mid}: {title}")
        print(f"  {summary}")

    macro_labels_v2[mid] = {
        "macro_cluster_id": mid,
        "title": title,
        "summary": summary,
        "coherent": coherent,
        "member_row_indices": [i for i, l in enumerate(final_labels) if l == mid],
    }

print(f"\nLabelled {len(macro_labels_v2)} macro-clusters.")

# ── Part B: place noise sub-clusters ─────────────────────────────────────────

macro_title_list = "\n".join(
    f"  macro_cluster_id={mid}: {macro_labels_v2[mid]['title']}"
    for mid in valid_macro_ids
)

# Build the working assignment array (copy of final_labels, noise slots to fill)
placement_labels = final_labels.copy()

noise_placement_log = {}

if noise_indices:
    print(f"\nPlacing {len(noise_indices)} noise sub-clusters via AI...\n")
    for idx in noise_indices:
        row       = sub_cluster_descriptors.iloc[idx]
        sc_id     = int(row["cluster_id"])
        cache_key = cluster_cache_key([f"noise_v2_{sc_id}"] + [f"ml_{m}" for m in sorted(valid_macro_ids)])

        if cache_key in label_cache:
            chosen = int(label_cache[cache_key]["macro_cluster_id"])
            print(f"  Sub-cluster [{sc_id}] '{row['title']}' → Macro {chosen} (cached)")
        else:
            prompt = f"""A sub-cluster of emails could not be automatically grouped.

Sub-cluster title: {row['title']}
Sub-cluster summary: {row['summary']}

Available macro-clusters:
{macro_title_list}

Which macro_cluster_id does this sub-cluster belong to?
Return ONLY valid JSON: {{"macro_cluster_id": <integer>}}"""

            response = ask_haiku(prompt, max_tokens=60)
            try:
                parsed = parse_json_response(response)
                chosen = int(parsed["macro_cluster_id"])
                if chosen not in valid_macro_ids:
                    raise ValueError(f"Unknown macro id {chosen}")
            except (json.JSONDecodeError, ValueError, KeyError):
                from collections import Counter
                chosen = Counter(
                    l for l in final_labels if l != -1
                ).most_common(1)[0][0]

            label_cache[cache_key] = {"macro_cluster_id": chosen}
            save_label_cache(label_cache)
            print(f"  Sub-cluster [{sc_id}] '{row['title']}' → Macro {chosen}: "
                  f"{macro_labels_v2[chosen]['title']}")

        placement_labels[idx] = chosen
        noise_placement_log[sc_id] = chosen
        macro_labels_v2[chosen]["member_row_indices"].append(idx)
else:
    print("\nNo noise sub-clusters to place.")

print("\nSTEP 3 complete.")

# ── Attach final assignments to descriptor dataframe ─────────────────────────

sub_cluster_descriptors["macro_cluster_id_v2"]    = placement_labels
sub_cluster_descriptors["macro_cluster_title_v2"] = sub_cluster_descriptors["macro_cluster_id_v2"].map(
    {mid: d["title"] for mid, d in macro_labels_v2.items()}
)
sub_cluster_descriptors["was_noise"] = [
    (i in noise_indices) for i in range(len(sub_cluster_descriptors))
]

In [ ]:
# --- Cell 14: STEP 4 — Save outputs and print hierarchy tree ---

import os
import json
import pandas as pd

os.makedirs("output/email_semantics", exist_ok=True)

# ── Build macro_cluster_labels_v2_haiku.csv ─────────────────────────────────────────

macro_rows = []
for mid in sorted(macro_labels_v2.keys()):
    d        = macro_labels_v2[mid]
    members  = sub_cluster_descriptors[sub_cluster_descriptors["macro_cluster_id_v2"] == mid]
    macro_rows.append({
        "macro_cluster_id":  mid,
        "title":             d["title"],
        "summary":           d["summary"],
        "sub_cluster_count": len(members),
        "total_emails":      int(members["email_count"].sum()),
        "sub_cluster_ids":   json.dumps(sorted(members["cluster_id"].tolist())),
        "sub_cluster_titles": json.dumps(members["title"].tolist()),
    })

macro_df_v2 = pd.DataFrame(macro_rows)
macro_df_v2.to_csv("output/email_semantics/macro_cluster_labels_v2_haiku.csv", index=False)

# ── Build sub_cluster_descriptors_with_macro_v2_haiku.csv ──────────────────────────

# Start from a copy to avoid mutating the working dataframe used by earlier cells.
desc_v2 = sub_cluster_descriptors.copy()

# Remove v1 macro columns before renaming v2 columns to final names.
drop_cols = [c for c in ["macro_cluster_id", "macro_cluster_title"] if c in desc_v2.columns]
if drop_cols:
    desc_v2 = desc_v2.drop(columns=drop_cols)

desc_v2 = desc_v2.rename(columns={
    "macro_cluster_id_v2": "macro_cluster_id",
    "macro_cluster_title_v2": "macro_cluster_title",
})

desc_v2.to_csv(
    "output/email_semantics/sub_cluster_descriptors_with_macro_v2_haiku.csv", index=False
)

# ── Print hierarchy tree ──────────────────────────────────────────────────────

print("=" * 70)
print("MACRO-CLUSTER HIERARCHY  (UMAP → HDBSCAN → AI v2)")
print("=" * 70)

for _, macro_row in macro_df_v2.sort_values("macro_cluster_id").iterrows():
    mid = macro_row["macro_cluster_id"]
    print(f"\nMACRO {mid}: {macro_row['title']}  ({macro_row['total_emails']} emails)")
    print(f"  {macro_row['summary']}")

    members = desc_v2[desc_v2["macro_cluster_id"] == mid].sort_values("cluster_id")
    for _, sr in members.iterrows():
        noise_tag = " [was noise]" if sr["was_noise"] else ""
        print(f"    • [{int(sr['cluster_id'])}] {sr['title']} ({int(sr['email_count'])} emails){noise_tag}")

print("\n" + "=" * 70)
print("Saved:")
print("  output/email_semantics/macro_cluster_labels_v2_haiku.csv")
print("  output/email_semantics/sub_cluster_descriptors_with_macro_v2_haiku.csv")

In [ ]:
# --- Cell 15: Export viz_data.json for the p5.js constellation visualisation ---
# Run this cell after Cell 14. Reads the in-memory dataframes produced by the
# pipeline (desc_v2, macro_df_v2, emails_df) and writes one JSON file that the
# browser loads directly — no CSV parsing needed on the front end.

import json, re, hashlib, os
from datetime import datetime, timezone

os.makedirs("output", exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def clean_subject(s):
    """Strip Re:/Fwd: prefixes and normalise for thread grouping."""
    s = re.sub(r'^(Re|Fwd|FW|RE|FWD)\s*:\s*', '', str(s), flags=re.IGNORECASE).strip()
    return s.lower() or "(no subject)"

def str_seed(s):
    """Stable integer seed from a string — used for deterministic blob shapes."""
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16) % 100000

# ── date normalisation ────────────────────────────────────────────────────────
# age = 1.0 → most recent email in the dataset
# age = 0.0 → oldest email in the dataset
# Emails without a parseable date get age = 0.5 (mid-range placeholder).

clustered_emails = emails_df[emails_df["cluster_id"] != -1].copy()

parsed_dates = pd.to_datetime(clustered_emails["date"], utc=True, errors="coerce")
valid_dates  = parsed_dates.dropna()

if len(valid_dates) >= 2:
    ts_min = valid_dates.min().timestamp()
    ts_max = valid_dates.max().timestamp()
    ts_range = ts_max - ts_min

    def date_to_age(date_str):
        try:
            ts = pd.to_datetime(date_str, utc=True).timestamp()
            return round((ts - ts_min) / ts_range, 4)
        except Exception:
            return 0.5
else:
    def date_to_age(date_str):
        return 0.5

# ── size normalisation ────────────────────────────────────────────────────────

ec_vals = desc_v2["email_count"].values
ec_min, ec_max = int(ec_vals.min()), int(ec_vals.max())

def norm_ec(v):
    return round((v - ec_min) / (ec_max - ec_min), 4) if ec_max != ec_min else 0.5

te_vals = macro_df_v2["total_emails"].values
te_min, te_max = int(te_vals.min()), int(te_vals.max())

def norm_te(v):
    return round((v - te_min) / (te_max - te_min), 4) if te_max != te_min else 0.5

# ── build per-cluster email lookup with thread grouping ───────────────────────

def build_threads_and_floating(cluster_id):
    rows = clustered_emails[clustered_emails["cluster_id"] == cluster_id]
    by_subject = {}
    for _, r in rows.iterrows():
        key = clean_subject(r["subject"])
        by_subject.setdefault(key, []).append(r)

    threads  = []
    floating = []
    for key, group in by_subject.items():
        nodes = [
            {
                "email_id": int(r.name),
                "age": date_to_age(r["date"]),
                "seed": str_seed(str(r["sender"]) + str(r["subject"]) + str(i)),
                "sender": str(r.get("sender", "")).strip(),
                "subject": str(r.get("subject", "")).strip(),
                "body_preview": str(r.get("body_preview", "")).strip(),
                "date_str": str(r.get("date", "")).strip() or "—",
                "size": str(r.get("size", "Unknown")).strip(),
            }
            for i, r in enumerate(group)
        ]
        if len(group) >= 2:
            threads.append({"nodes": nodes})
        else:
            floating.extend(nodes)

    return threads, floating

# ── assemble MACROS structure ─────────────────────────────────────────────────

macros_out = []

for _, macro_row in macro_df_v2.sort_values("macro_cluster_id").iterrows():
    mid = int(macro_row["macro_cluster_id"])

    sub_rows = desc_v2[desc_v2["macro_cluster_id"] == mid].sort_values(
        "email_count", ascending=False
    )

    sub_clusters = []
    for _, sr in sub_rows.iterrows():
        cid     = int(sr["cluster_id"])
        threads, floating = build_threads_and_floating(cid)

        # Sub-cluster age = average of its email ages (fallback to cohesion proxy)
        all_nodes = [n for t in threads for n in t["nodes"]] + floating
        if all_nodes:
            sub_age = round(sum(n["age"] for n in all_nodes) / len(all_nodes), 4)
        else:
            sub_age = round(float(sr["cohesion_score"]), 4)

        sub_clusters.append({
            "id":       cid,
            "title":    sr["title"],
            "summary":  sr["summary"],
            "size":     norm_ec(int(sr["email_count"])),
            "age":      sub_age,
            "seed":     cid,
            "threads":  threads,
            "floating": floating,
        })

    macro_age = (
        round(sum(s["age"] for s in sub_clusters) / len(sub_clusters), 4)
        if sub_clusters else 0.5
    )

    macros_out.append({
        "id":          mid,
        "title":       macro_row["title"],
        "summary":     macro_row["summary"],
        "size":        norm_te(int(macro_row["total_emails"])),
        "age":         macro_age,
        "subClusters": sub_clusters,
    })

# ── write viz_data.json (lean, no full bodies) ────────────────────────────────

output = {
    "generated_at":    datetime.now(timezone.utc).isoformat(),
    "email_count":     int(len(clustered_emails)),
    "macro_count":     len(macros_out),
    "sub_cluster_count": sum(len(m["subClusters"]) for m in macros_out),
    "macros":          macros_out,
}

out_path = "output/viz_data_haiku.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

size_kb = os.path.getsize(out_path) / 1024
print(f"Wrote {out_path}  ({size_kb:.1f} KB)")
print(f"  {output['macro_count']} macro-clusters")
print(f"  {output['sub_cluster_count']} sub-clusters")
print(f"  {output['email_count']} emails (clustered)")
if len(valid_dates) >= 2:
    print(f"  date range: {valid_dates.min().date()} → {valid_dates.max().date()}")
else:
    print("  no parseable dates found — age values set to 0.5")

# ── write emails_full_contents.json (on-demand lookup for full bodies) ────────

emails_lookup = {i: e for i, e in enumerate(emails)}
emails_list = []
for idx, row in clustered_emails.iterrows():
    email_idx = int(idx)
    full_email = emails_lookup.get(email_idx, {})
    subject_full = str(full_email.get("subject", row.get("subject", ""))).strip()
    body_full = str(full_email.get("body", row.get("body", ""))).strip()
    emails_list.append({
        "id": email_idx,
        "sender": str(full_email.get("sender", row.get("sender", ""))).strip(),
        "subject": subject_full,
        "subject_full": subject_full,
        "date": str(full_email.get("date", row.get("date", ""))).strip(),
        "body": body_full,
        "body_preview": str(row.get("body_preview", "")).strip(),
        "size": str(row.get("size", "Unknown")).strip(),
    })

emails_output = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "email_count": len(emails_list),
    "emails": emails_list,
}

emails_path = "output/emails_full_contents.json"
with open(emails_path, "w") as f:
    json.dump(emails_output, f, indent=2)

size_kb_emails = os.path.getsize(emails_path) / 1024
print(f"\nWrote {emails_path}  ({size_kb_emails:.1f} KB)")
print(f"  {emails_output['email_count']} emails with full bodies")

# Quick sanity check: show longest body length and tail snippet
if emails_list:
    max_body = max(emails_list, key=lambda x: len(x.get("body", "")))
    print(f"Longest body length: {len(max_body.get('body', ''))}")
    print(f"Body tail (last 200 chars): {max_body.get('body', '')[-200:]}")

In [ ]:
# --- Cell 16: Galaxy clustering on macro centroids ---
# Groups macro-clusters into 4-8 galaxies using HDBSCAN on the same UMAP space.
# Uses the macro centroid positions (already computed in umap_matrix / final_labels).
# Noise macros are assigned to nearest galaxy by cosine similarity.

import numpy as np
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity

# ── Diagnostic: show what Cell 12 / 12c produced ─────────────────────────────
valid_macro_ids = sorted(set(final_labels) - {-1})
n_noise = int((np.array(final_labels) == -1).sum())
print(f"Cell 12/12c result: {len(valid_macro_ids)} macros, "
      f"{n_noise}/{len(final_labels)} sub-clusters are noise")

# ── Fallback: if HDBSCAN found 0 macros, split sub-clusters by KMeans ─────────
if not valid_macro_ids:
    print("WARNING: No macro-clusters — HDBSCAN found only noise.")
    print("Falling back: KMeans into 10 macro groups on raw UMAP space.")
    from sklearn.cluster import KMeans
    n_fallback = min(10, len(final_labels))
    km = KMeans(n_clusters=n_fallback, random_state=42, n_init="auto")
    km_labels = km.fit_predict(final_matrix)
    final_labels = km_labels
    valid_macro_ids = sorted(set(final_labels) - {-1})
    print(f"KMeans fallback produced {len(valid_macro_ids)} macro groups")

# ── Build matrix of macro centroids in UMAP space ────────────────────────────
macro_umap_rows = []
macro_idx_map   = {}    # position in matrix → macro_id

for pos, mid in enumerate(valid_macro_ids):
    sc_indices = [i for i, l in enumerate(final_labels) if l == mid]
    if not sc_indices:
        continue
    avg_umap   = final_matrix[sc_indices].mean(axis=0)
    macro_umap_rows.append(avg_umap)
    macro_idx_map[pos] = mid

macro_umap_matrix = np.array(macro_umap_rows)
n_macros = len(macro_idx_map)

print(f"Clustering {n_macros} macros into galaxies...")

# Target: proportional so it works at any dataset scale
target_lo = max(2, n_macros // 5)
target_hi = max(4, n_macros // 2)
print(f'  Galaxy target: {target_lo}–{target_hi}')
best_labels = None

for mcs in range(max(2, n_macros // 8), max(3, n_macros // 3) + 1):
    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=1, metric="euclidean")
    labels = cl.fit_predict(macro_umap_matrix)
    n_g = len(set(labels) - {-1})
    print(f"  min_cluster_size={mcs} → {n_g} galaxies, {(labels==-1).sum()} noise macros")
    if target_lo <= n_g <= target_hi:
        best_labels = labels.copy()
        break

if best_labels is None:
    print("No result in [4,8] — using last result")
    best_labels = labels.copy()

galaxy_ids = sorted(set(best_labels) - {-1})

# ── Fallback: if HDBSCAN found no galaxy clusters, use KMeans ────────────────
if not galaxy_ids:
    print("HDBSCAN found no galaxy clusters — falling back to KMeans.")
    from sklearn.cluster import KMeans
    n_gal = min(max(2, n_macros // 3), 8)
    km = KMeans(n_clusters=n_gal, random_state=42, n_init="auto")
    best_labels = km.fit_predict(macro_umap_matrix)
    galaxy_ids  = sorted(set(best_labels))  # KMeans has no noise
    print(f"KMeans produced {len(galaxy_ids)} galaxies")

# Assign noise macros to nearest galaxy centroid (only needed for HDBSCAN results)
galaxy_centroids = np.array([
    macro_umap_matrix[best_labels == g].mean(axis=0) for g in galaxy_ids
])

galaxy_labels = best_labels.copy()
for i, lbl in enumerate(best_labels):
    if lbl == -1:
        sims = cosine_similarity(macro_umap_matrix[i].reshape(1,-1), galaxy_centroids)[0]
        galaxy_labels[i] = galaxy_ids[int(np.argmax(sims))]

# Map macro_id → galaxy_id
macro_to_galaxy = {macro_idx_map[pos]: int(galaxy_labels[pos]) for pos in range(n_macros)}

print(f"\nGalaxy assignments:")
for g in galaxy_ids:
    members = [macro_idx_map[p] for p, gl in enumerate(galaxy_labels) if gl == g]
    print(f"  Galaxy {g}: macro ids {members}")


In [ ]:
# --- Cell 17: Label galaxies via Haiku ---
# Each galaxy receives a high-level domain label from its macro titles/summaries.
# Privacy: only macro titles (already abstracted) are sent — no raw email content.

import json, re

galaxy_labels_v2 = {}

print("Labelling galaxies...\n")
for g in galaxy_ids:
    member_macro_ids = [macro_idx_map[p] for p, gl in enumerate(galaxy_labels) if gl == g]
    # Pull macro titles and summaries from macro_labels_v2
    macro_descs = "\n".join(
        f"- {macro_labels_v2[mid]['title']}: {macro_labels_v2[mid]['summary'][:120]}"
        for mid in member_macro_ids if mid in macro_labels_v2
    )

    prompt = f"""You are labelling a high-level domain region of someone's inbox.
This region contains the following topic groups:
{macro_descs}

Give this region:
1. A domain-level title (max 4 words — broad category like "Energy Trading", "Property & Finance", "Team & HR")
2. One sentence describing the common domain thread

Return ONLY valid JSON:
{{"title": "...", "summary": "..."}}"""

    response = ask_haiku(prompt, max_tokens=200)
    clean = re.sub(r"```json|```", "", response).strip()
    start = clean.find("{"); end = clean.rfind("}") + 1
    try:
        parsed  = json.loads(clean[start:end])
        title   = str(parsed.get("title", "Unclassified")).strip()[:40]
        summary = str(parsed.get("summary", "")).strip()
    except Exception:
        title   = f"Galaxy {g}"
        summary = ""

    galaxy_labels_v2[g] = {"galaxy_id": g, "title": title, "summary": summary,
                             "macro_ids": member_macro_ids}
    print(f"Galaxy {g}: {title}")
    print(f"  {summary}\n")


In [ ]:
# --- Cell 18: Export split files for Flask lazy-loading ---
import numpy as _np

class _NumpyEncoder(json.JSONEncoder):
    """Convert numpy scalars to native Python types for JSON serialisation."""
    def default(self, obj):
        if isinstance(obj, _np.integer): return int(obj)
        if isinstance(obj, _np.floating): return float(obj)
        if isinstance(obj, _np.ndarray): return obj.tolist()
        return super().default(obj)

#
# Outputs (do NOT overwrite viz_data.json — presentation version stays intact):
#   output/cluster_structure.json      full galaxy→macro→sub hierarchy, NO email bodies
#   output/emails_meta/sub_{id}.json   per-sub-cluster email metadata (loaded on cluster open)
#   output/email_body/email_{id}.json  individual email body (loaded on email click)

import json, os, hashlib, re, pandas as pd
from datetime import datetime, timezone

os.makedirs("../web/output/emails_meta", exist_ok=True)
os.makedirs("../web/output/email_body",  exist_ok=True)

# ── helpers (same as Cell 15) ─────────────────────────────────────────────────
def clean_subject(s):
    return re.sub(r'^(Re|Fwd|FW|RE|FWD)\s*:\s*', '', str(s), flags=re.IGNORECASE).strip().lower() or "(no subject)"

def str_seed(s):
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16) % 100000

clustered_emails = emails_df[emails_df["cluster_id"] != -1].copy()
parsed_dates = pd.to_datetime(clustered_emails["date"], utc=True, errors="coerce")
valid_dates  = parsed_dates.dropna()

if len(valid_dates) >= 2:
    ts_min, ts_max = valid_dates.min().timestamp(), valid_dates.max().timestamp()
    ts_range = ts_max - ts_min
    def date_to_age(d):
        try: return round((pd.to_datetime(d, utc=True).timestamp() - ts_min) / ts_range, 4)
        except: return 0.5
else:
    def date_to_age(d): return 0.5

ec_vals = desc_v2["email_count"].values
ec_min, ec_max = int(ec_vals.min()), int(ec_vals.max())
def norm_ec(v): return round((v - ec_min) / (ec_max - ec_min), 4) if ec_max != ec_min else 0.5

te_vals = macro_df_v2["total_emails"].values
te_min, te_max = int(te_vals.min()), int(te_vals.max())
def norm_te(v): return round((v - te_min) / (te_max - te_min), 4) if te_max != te_min else 0.5

# ── Write per-email body files ────────────────────────────────────────────────
emails_lookup = {i: e for i, e in enumerate(emails)}
written_bodies = 0
for idx, row in clustered_emails.iterrows():
    eid = int(idx)
    full = emails_lookup.get(eid, {})
    body = {
        "id":      eid,
        "sender":  str(full.get("sender", "")).strip(),
        "subject": str(full.get("subject", row.get("subject",""))).strip(),
        "date":    str(full.get("date",   row.get("date",""))).strip(),
        "body":    str(full.get("body",   "")).strip(),
    }
    with open(f"../web/output/email_body/email_{eid}.json", "w") as f:
        json.dump(body, f, cls=_NumpyEncoder)
    written_bodies += 1

print(f"Wrote {written_bodies} email_body files")

# ── Write per-sub-cluster email metadata files ────────────────────────────────
written_meta = 0
for _, sr in desc_v2.iterrows():
    cid  = int(sr["cluster_id"])
    rows = clustered_emails[clustered_emails["cluster_id"] == cid]
    # Build email lookup for size/attachment fields (emails list → dict keyed by index)
    by_subject = {}
    for _, r in rows.iterrows():
        key = clean_subject(r["subject"])
        by_subject.setdefault(key, []).append(r)

    # Full email lookup for size/attachment data
    _email_by_idx = {i: e for i, e in enumerate(emails)}

    threads, floating = [], []
    for key, group in by_subject.items():
        nodes = [{"email_id": int(r.name), "age": date_to_age(r["date"]),
                  "seed": str_seed(str(r["sender"]) + str(r["subject"]) + str(i)),
                  "sender": str(r.get("sender","")).strip(),
                  "subject": str(r.get("subject","")).strip(),
                  "body_preview": str(r.get("body_preview","")).strip(),
                  "date_str": str(r.get("date","")).strip() or "—",
                  "size_kb": _email_by_idx.get(int(r.name), {}).get("simulated_size_kb", 80),
                  "attachment": _email_by_idx.get(int(r.name), {}).get("attachment_type", None)}
                 for i, r in enumerate(group)]
        if len(group) >= 2: threads.append({"nodes": nodes})
        else:               floating.extend(nodes)

    with open(f"../web/output/emails_meta/sub_{cid}.json", "w") as f:
        json.dump({"threads": threads, "floating": floating}, f, cls=_NumpyEncoder)
    written_meta += 1

print(f"Wrote {written_meta} emails_meta files")

# ── Build cluster_structure.json (galaxy→macro→sub, no email content) ─────────
galaxies_out = []
for g in sorted(galaxy_labels_v2.keys()):
    gdata      = galaxy_labels_v2[g]
    macro_ids  = gdata["macro_ids"]

    macros_out = []
    for mid in macro_ids:
        if mid not in macro_labels_v2: continue
        mdata = macro_labels_v2[mid]
        sub_rows = desc_v2[desc_v2["macro_cluster_id"] == mid].sort_values("email_count", ascending=False)

        subs_out = []
        for _, sr in sub_rows.iterrows():
            cid = int(sr["cluster_id"])
            all_rows = clustered_emails[clustered_emails["cluster_id"] == cid]
            ages = [date_to_age(r["date"]) for _, r in all_rows.iterrows()]
            sub_age = round(sum(ages)/len(ages), 4) if ages else 0.5
            # Data size: sum simulated_size_kb for all emails in this sub-cluster
            sub_email_indices = rows.index.tolist()
            sub_data_kb = sum(
                _email_by_idx.get(int(idx), {}).get("simulated_size_kb", 80)
                for idx in sub_email_indices
            )
            subs_out.append({
                "id":           cid,
                "title":        sr["title"],
                "summary":      sr["summary"],
                "size":         norm_ec(int(sr["email_count"])),  # replaced below with composite
                "age":          sub_age,
                "seed":         cid,
                "email_count":  int(sr["email_count"]),
                "data_size_kb": round(sub_data_kb, 1),
            })

        total_emails  = int(sub_rows["email_count"].sum())
        macro_age     = round(sum(s["age"] for s in subs_out)/len(subs_out), 4) if subs_out else 0.5
        macro_data_kb = sum(s["data_size_kb"] for s in subs_out)
        macros_out.append({
            "id":           mid,
            "title":        mdata["title"],
            "summary":      mdata["summary"],
            "size":         norm_te(total_emails),   # replaced with composite below
            "age":          macro_age,
            "subClusters":  subs_out,
            "email_count":  total_emails,
            "data_size_kb": round(macro_data_kb, 1),
        })

    g_total      = sum(len(m["subClusters"]) for m in macros_out)
    g_emails     = sum(s["email_count"] for m in macros_out for s in m["subClusters"])
    g_size       = round(g_emails / (len(clustered_emails) or 1), 4)
    g_age        = round(sum(m["age"] for m in macros_out)/len(macros_out), 4) if macros_out else 0.5
    g_data_kb    = sum(m["data_size_kb"] for m in macros_out)

    galaxies_out.append({
        "id":           g,
        "title":        gdata["title"],
        "summary":      gdata["summary"],
        "size":         g_size,
        "age":          g_age,
        "macros":       macros_out,
        "email_count":  g_emails,
        "data_size_kb": round(g_data_kb, 1),
    })

# ── Apply composite size (count + data volume) across all levels ──────────────
all_subs  = [s for g in galaxies_out for m in g["macros"] for s in m["subClusters"]]
all_macros = [m for g in galaxies_out for m in g["macros"]]

if all_subs:
    sc_count_vals = [s["email_count"] for s in all_subs]
    sc_data_vals  = [s["data_size_kb"] for s in all_subs]
    sc_cmin, sc_cmax = min(sc_count_vals), max(sc_count_vals)
    sc_dmin, sc_dmax = min(sc_data_vals),  max(sc_data_vals)
    for s in all_subs:
        nc = (s["email_count"]  - sc_cmin) / (sc_cmax - sc_cmin) if sc_cmax != sc_cmin else 0.5
        nd = (s["data_size_kb"] - sc_dmin) / (sc_dmax - sc_dmin) if sc_dmax != sc_dmin else 0.5
        s["size"] = round(0.5 * nc + 0.5 * nd, 4)

if all_macros:
    mc_count_vals = [m["email_count"]  for m in all_macros]
    mc_data_vals  = [m["data_size_kb"] for m in all_macros]
    mc_cmin, mc_cmax = min(mc_count_vals), max(mc_count_vals)
    mc_dmin, mc_dmax = min(mc_data_vals),  max(mc_data_vals)
    for m in all_macros:
        nc = (m["email_count"]  - mc_cmin) / (mc_cmax - mc_cmin) if mc_cmax != mc_cmin else 0.5
        nd = (m["data_size_kb"] - mc_dmin) / (mc_dmax - mc_dmin) if mc_dmax != mc_dmin else 0.5
        m["size"] = round(0.5 * nc + 0.5 * nd, 4)

total_inbox_kb = sum(e.get("simulated_size_kb", 80) for e in emails)

structure = {
    "generated_at":      datetime.now(timezone.utc).isoformat(),
    "email_count":       int(len(clustered_emails)),
    "galaxy_count":      len(galaxies_out),
    "macro_count":       sum(len(g["macros"]) for g in galaxies_out),
    "sub_cluster_count": sum(len(m["subClusters"]) for g in galaxies_out for m in g["macros"]),
    "total_data_size_kb": round(total_inbox_kb, 1),
    "galaxies":          galaxies_out,
}

out_path = "../web/output/cluster_structure.json"
with open(out_path, "w") as f:
    json.dump(structure, f, indent=2, cls=_NumpyEncoder)

kb = os.path.getsize(out_path) // 1024
print(f"\nWrote {out_path}  ({kb} KB)")
print(f"  {structure['galaxy_count']} galaxies")
print(f"  {structure['macro_count']} macros")
print(f"  {structure['sub_cluster_count']} sub-clusters")
print(f"  {structure['email_count']} emails")
print(f"\nviz_data.json NOT modified — presentation version is safe.")
